# Comparativo global: MASTER, TFB, RandomTopJ e benchmarks

Este notebook consolida os JSONs de resultados e, quando houver `sinais.csv` na mesma pasta, incorpora as AUCs direcionais.

**Ponto importante de interpretação:** TFB e MASTER podem ter `pred_len` com significado diferente. Por isso, a comparação principal deve ser feita por `janela_trading`/`horizonte_comparavel` (`k` de rebalanceamento), não apenas por `pred_len`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'utils':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.comparativo_metricas_global import comparar_global, METRICAS_INTERESSE

OUT = ROOT / 'simulacoes' / 'comparativo_global_master_tfb'
OUT


In [ ]:
dfs = comparar_global(
    base_dir=ROOT,
    output_dir='simulacoes/comparativo_global_master_tfb',
    top_n=30,
)

metricas = dfs['metricas']
resumo = dfs['resumo']
top_configs = dfs['top_configs']
ranking = dfs['ranking']

print(f'Linhas carregadas: {len(metricas)}')
print(f'Arquivos salvos em: {OUT}')
metricas.head()


In [ ]:
# Cobertura por grupo
if not metricas.empty:
    display(metricas.groupby('grupo').size().rename('n_resultados').reset_index())
    display(metricas.groupby(['grupo', 'modelo']).size().rename('n_resultados').reset_index().sort_values(['grupo', 'n_resultados'], ascending=[True, False]).head(50))


In [ ]:
# Tabela principal das métricas de interesse
cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', *METRICAS_INTERESSE, 'json_path']
tabela = metricas[[c for c in cols if c in metricas.columns]].copy()
tabela.sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last').head(50)


In [ ]:
# Melhores configurações por métrica
top_configs.head(80)


In [ ]:
# Ranking médio simples nas métricas de interesse: quanto menor, melhor.
ranking_cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', 'rank_medio_metricas_interesse', *METRICAS_INTERESSE, 'json_path']
ranking[[c for c in ranking_cols if c in ranking.columns]].head(50)


In [ ]:
# Resumo agregado por grupo/modelo/janela de trading
resumo.sort_values(['janela_trading', 'mean_spearman_ic_mean'], ascending=[True, False], na_position='last').head(80)


In [ ]:
# Comparação direta por janela de trading e modelo, usando medianas
if not metricas.empty:
    comp = (
        metricas
        .groupby(['janela_trading', 'grupo', 'modelo'], dropna=False)[[c for c in METRICAS_INTERESSE if c in metricas.columns]]
        .median()
        .reset_index()
        .sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last')
    )
    display(comp.head(100))


## Arquivos gerados

- `metricas_global_long.csv`: base longa, uma linha por JSON encontrado.
- `tabela_metricas_interesse.csv`: somente as métricas centrais.
- `resumo_por_modelo.csv`: média/mediana/desvio/contagem por grupo, dataset, modelo e janela de trading.
- `top_configs_por_metrica.csv`: melhores configurações por métrica.
- `ranking_global.csv`: ranking médio simples das métricas de interesse.
